In [ ]:
import pandas as pd
import numpy as np

In [5]:
## Read in encoded_fourth_downs.csv.gz
fourth_downs = pd.read_csv("../encoded_fourth_downs.csv.gz", compression="gzip")

In [6]:
fourth_downs.head()

,is_fourth_down,is_hard_count,week,yardline_100,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,drive,sp,qtr,...,playcaller_Shane Steichen,playcaller_Shane Waldron,playcaller_Steve Sarkisian,playcaller_Thomas Brown,playcaller_Tim Kelly,playcaller_Todd Downing,playcaller_Todd Haley,playcaller_Todd Monken,playcaller_Zac Robinson,playcaller_Zac Taylor
0,0,1,11,60.0,826,1726,3526,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,0,1,5,72.0,814,1714,3514,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,0,1,0,67.0,861,1761,3561,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,1,0,17,83.0,787,1687,3487,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,1,0,16,76.0,819,1719,3519,0,0,0,...,False,False,False,False,False,False,False,False,False,False


In [9]:
## Build model to predict EPA on fourth downs
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
X = fourth_downs.drop(columns=["epa"])
y = fourth_downs["epa"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 0.043972507518403925


In [10]:
# View MSE vs. Descriptive Statistics of EPA
print(f"Mean EPA: {y.mean()}")
print(f"Median EPA: {y.median()}")
print(f"Standard Deviation of EPA: {y.std()}")
print(f"Mean Squared Error: {mse}")

Mean EPA: -0.016282519941457477
Median EPA: 0.0128834107890725
Standard Deviation of EPA: 1.6430722862710156
Mean Squared Error: 0.043972507518403925


In [11]:
## Test for Overfitting
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f"Training R^2 Score: {train_score}")
print(f"Testing R^2 Score: {test_score}")

Training R^2 Score: 0.9979869176764964
Testing R^2 Score: 0.9836868895071229


In [18]:
## Further overfitting test: Compare feature importances to descriptive statistics of features
feature_importances = pd.Series(model.feature_importances_, index=X.columns)
feature_stats = X.describe().T
feature_stats["importance"] = feature_importances
# view problematic features with very low importance
print(feature_stats[feature_stats["importance"] < 0.01][["mean", "std", "importance"]])

                                    mean         std  importance
is_fourth_down                  0.954646    0.208081    0.000339
is_hard_count                   0.053790    0.225605    0.000605
week                            9.976962    6.765638    0.000273
yardline_100                   46.903359   25.946088    0.001656
quarter_seconds_remaining     399.730638  266.554227    0.000827
...                                  ...         ...         ...
td_is_posteam                   0.015556    0.123753    0.000700
penalty_is_posteam              0.050797    0.219586    0.000579
return_is_posteam               0.000000    0.000000    0.000000
fumbled_1_is_posteam            0.005108    0.071289    0.000026
fumble_recovery_1_is_posteam    0.007636    0.087053    0.000341

[141 rows x 3 columns]
